# Лекция 1. Первое знакомство с данными

Сегодня мы четыре раза сделаем одно и то же движение: **зададим вопрос к данным и получим на него ответ кодом**.

1. **Загрузим** таблицу с отчётностью американских банков.
2. **Посмотрим** на неё: что внутри, что с ней не так, что с чем связано.
3. Объясним прибыльность банка **линейной регрессией** — и увидим, чего простая регрессия сказать не может.
4. Сравним регрессию с **методом ближайших соседей**: моделью, которая не предполагает никакой формулы.

Как работать с ноутбуком: ячейки запускаются **сверху вниз**, по `Shift+Enter`. Если пропустить ячейку, следующая, скорее всего, сломается: переменные создаются по порядку.

## 0. Подготовка

Эта ячейка есть в каждом ноутбуке курса. Она подключает библиотеки и говорит, откуда брать данные. Запустите её первой.

**Библиотека** — набор чужого готового кода, который мы берём и используем. `import pandas as pd` означает «подключи библиотеку `pandas` и дальше называй её коротко `pd`».

In [ ]:
# Библиотеки. Импортируем один раз в начале — дальше они доступны во всём ноутбуке.
import numpy as np                 # массивы и математика
import pandas as pd                # таблицы: главный инструмент курса
import matplotlib.pyplot as plt    # рисование графиков
import seaborn as sns              # графики покрасивее, надстройка над matplotlib

# Откуда берём данные: файлы лежат в репозитории курса, читаем их прямо по ссылке.
DATA = "https://raw.githubusercontent.com/nvvoitov/da_intro/main/data/"

# Фиксируем случайность: одно и то же число -> одинаковый результат у всех в аудитории.
SEED = 20260101

# Косметика: как выглядят графики и таблицы. Вникать не обязательно.
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (7, 4)     # размер картинки по умолчанию
pd.set_option("display.max_columns", 30)    # показывать до 30 колонок таблицы

## 1. Импорт данных

Данные о банках США собирает FDIC — американский аналог АСВ. Он публикует **квартальную отчётность каждого банка** и список банкротств. Мы взяли оттуда 2001–2026 годы: все банки, которые за это время упали, плюс случайную выборку выживших.

Файл лежит в формате **parquet**. Это как csv, только данные хранятся не текстом, а в двоичном виде: файл меньше, читается быстрее, и типы колонок (число, дата, текст) сохраняются сами. Читаем его командой `pd.read_parquet`.

Денежные величины — **в тысячах долларов**.

In [ ]:
# Создаём переменную panel, в которой будет лежать вся таблица.
# Знак + склеивает адрес папки и имя файла в одну ссылку.
panel = pd.read_parquet(DATA + "banks_panel.parquet")

Ячейка выше ничего не напечатала — она просто положила таблицу в переменную. Чтобы что-то увидеть, надо об этом попросить.

Начнём с размера: `.shape` возвращает пару чисел — **строк** и **колонок**.

In [ ]:
# .shape — это не команда, а свойство таблицы: пара (строки, колонки)
print("строк и колонок:", panel.shape)

# len() считает строки, .nunique() — сколько разных значений в колонке
print("строк (банк-квартал):", len(panel))
print("разных банков:", panel["cert"].nunique())

# .min() и .max() по колонке с датой отчётности
print("период:", panel["repdte"].min().date(), "..", panel["repdte"].max().date())

Обратите внимание: строк 152 тысячи, а банков — 2.5 тысячи. Один банк встречается в таблице много раз, по разу на каждый квартал.

**Одна строка = один банк в один квартал.** Это самая важная фраза про эти данные, и к ней мы будем возвращаться весь курс.

Теперь посмотрим на несколько строк глазами. `.head()` показывает первые 5.

In [ ]:
# В двойных квадратных скобках — список колонок, которые хотим увидеть.
# Без него напечатались бы все 42 колонки, и читать было бы неудобно.
panel[["cert", "name", "state", "repdte", "assets", "equity", "net_income"]].head()

Что в колонках:

* `cert` — номер лицензии FDIC. Это **ключ банка**: имена меняются, номер — нет.
* `repdte` — дата отчётности (конец квартала).
* `assets`, `equity`, `net_income` — активы, капитал, прибыль, в тысячах долларов.

Работать с сырыми суммами неудобно: банк на 300 миллионов и банк на 300 миллиардов несравнимы. Поэтому в таблице уже посчитаны **отношения** (ratio) — суммы, поделённые на масштаб банка. Их имена начинаются с буквы блока CAMELS:

| Начало имени | Что меряем | Пример |
|---|---|---|
| `c_` | капитал | `c_equity_assets` = капитал / активы |
| `a_` | качество активов | `a_npl_gross_loans` = просрочка / кредиты |
| `m_` | эффективность | `m_cost_income` = расходы / процентный доход |
| `e_` | прибыльность | `e_roa` = прибыль / активы |
| `l_` | ликвидность | `l_loans_deposits` = кредиты / депозиты |
| `s_` | рыночный риск | `s_securities_assets` = ценные бумаги / активы |

Полный список колонок — командой `.columns`.

In [ ]:
# list() превращает результат в обычный список, его удобнее читать
print(list(panel.columns))

## 2. Как «посмотреть» на данные

У нас 152 тысячи строк. Открыть их глазами нельзя, поэтому смотреть будем **сводками и картинками**. Порядок почти всегда один и тот же:

1. что за колонки и есть ли пропуски — `.info()`;
2. числовые сводки — `.describe()`;
3. распределение каждой переменной по отдельности — гистограммы;
4. связи между переменными — диаграммы рассеяния и корреляции.

### 2.1 Что за колонки

In [ ]:
# .info() печатает: имя колонки, сколько непустых значений, тип данных
panel.info()

Читаем вывод:

* **`Non-Null Count`** — сколько значений заполнено. Где меньше 152 152, там **пропуски**. У `fail_date` их почти везде: она заполнена только у банков, которые действительно упали.
* **`Dtype`** — тип. `float64`/`float32` — число, `datetime64` — дата, `bool` — да/нет, `string`/`category` — текст.

Тип важен: с датой можно сравнивать «раньше/позже», с текстом — нет.

### 2.2 Числовые сводки

In [ ]:
# .describe() считает count, среднее, стандартное отклонение, min, квартили, max.
# .T переворачивает таблицу (колонки становятся строками) — так читать удобнее.
# .round(3) округляет до 3 знаков.
panel[["assets", "e_roa", "c_equity_assets", "a_npl_gross_loans",
       "m_cost_income", "l_loans_deposits"]].describe().T.round(3)

Смотрите на края. `m_cost_income` — это расходы, делённые на процентный доход; у нормального банка это 0.5–1.0. А в таблице минимум −19 496 и максимум 20 653.

Это не ошибка FDIC. Это банки, у которых процентный доход почти ноль (знаменатель), — трастовые компании, карточные банки, банки в процессе ликвидации. Деление на «почти ноль» даёт «почти бесконечность».

**Такие значения ломают и графики, и модели.** Сначала увидим, как именно.

### 2.3 Распределения

In [ ]:
# Гистограмма: по горизонтали — значение, по вертикали — сколько банк-кварталов попало в этот интервал.
sns.histplot(data=panel, x="m_cost_income", bins=50)
plt.title("Расходы / процентный доход: как есть")
plt.show()

График бесполезен: один столбик у нуля и пустота на всю ширину. Виноваты те самые выбросы — из-за них ось растянута на 40 тысяч, а все настоящие банки слиплись в одну палку.

Посмотрим на ту же переменную в разумных границах.

In [ ]:
# binrange=(0, 2) — рисуем только интервал от 0 до 2, остальное игнорируем
sns.histplot(data=panel, x="m_cost_income", bins=50, binrange=(0, 2))
plt.title("Расходы / процентный доход: в разумных границах")
plt.show()

Вот теперь видно распределение: горб около 0.8, длинный хвост вправо.

Гистограмму хочется построить не для одной колонки, а для нескольких. Копировать три строки шесть раз — плохая идея (шесть мест, где можно ошибиться). Для этого есть **цикл `for`**: он выполняет одни и те же действия для каждого элемента списка.

In [ ]:
# Список колонок и границ для каждой: (имя колонки, от, до)
COLS = [("c_equity_assets",   0, 0.3),
        ("a_npl_gross_loans", 0, 0.1),
        ("m_cost_income",     0, 2.0),
        ("e_roa",         -0.03, 0.03),
        ("l_loans_deposits",  0, 1.5),
        ("e_nim",             0, 0.08)]

# plt.subplots(2, 3) создаёт сетку 2 строки на 3 колонки. axes — массив из шести "холстов".
fig, axes = plt.subplots(2, 3, figsize=(13, 6))

# .ravel() вытягивает сетку 2x3 в один список из 6 элементов, чтобы идти по ним подряд.
# zip() ставит в пару каждую колонку и её холст.
for (col, lo, hi), ax in zip(COLS, axes.ravel()):
    sns.histplot(data=panel, x=col, bins=40, binrange=(lo, hi), ax=ax)
    ax.set_title(col)          # заголовок каждой картинки
    ax.set_xlabel("")          # подпись оси не нужна, она дублирует заголовок

plt.tight_layout()             # чтобы подписи не наезжали друг на друга
plt.show()

Шесть картинок написаны один раз. Захотим седьмую — добавим строчку в `COLS`, код менять не надо.

### 2.4 Чистка

Мы дважды спрятали выбросы **на картинке**. Но модель их не спрячет: она честно посчитает и банк с `m_cost_income = 20 653`. Значит, решение надо принять **явно** — и записать его так, чтобы его было видно и можно было оспорить.

In [ ]:
# Оставляем строки, где выполнены ВСЕ условия сразу.
# & означает "и". Каждое условие в своих скобках — это требование pandas.
banks = panel[
    (panel["assets"] > 50_000)                          # больше 50 млн $: не микроструктура
    & panel["e_roa"].between(-0.10, 0.10)               # ROA от -10% до +10% годовых
    & panel["m_cost_income"].between(0, 2)              # расходы от 0 до 2 процентных доходов
    & panel["a_nco_rate"].between(0, 0.05)              # списания от 0 до 5% кредитов
    & panel["c_equity_assets"].between(0.02, 0.30)      # капитал от 2% до 30% активов
].copy()                                                # .copy() — работаем с копией, не с куском исходной таблицы

print("было строк:  ", len(panel))
print("осталось:    ", len(banks), f"({len(banks) / len(panel):.0%})")
print("банков:      ", banks["cert"].nunique())

Отброшено больше трети строк. **Это не безобидно.** Мы только что поменяли ту совокупность, о которой будем делать выводы: дальше все наши утверждения относятся к «обычным работающим банкам», а не ко всем учреждениям с лицензией FDIC.

Проговаривать такие решения вслух — часть профессии. Молча их делать — способ получить неприятный вопрос на защите.

### 2.5 Связи между переменными

Одна переменная — гистограмма. Две переменные — **диаграмма рассеяния** (scatter): каждая точка это один банк-квартал, по горизонтали одно, по вертикали другое.

152 тысячи точек рисовать не нужно: картинка превратится в кляксу, а ждать придётся долго. Возьмём случайные 3000 — `.sample()`.

In [ ]:
# random_state=SEED — та самая фиксация случайности: выборка будет одна и та же при каждом запуске
show = banks.sample(3000, random_state=SEED)

# alpha=0.3 — прозрачность точек: там, где их много, пятно темнее
sns.scatterplot(data=show, x="m_cost_income", y="e_roa", alpha=0.3, s=12)
plt.title("Чем больше расходов на рубль дохода, тем ниже прибыльность")
plt.xlabel("расходы / процентный доход")
plt.ylabel("ROA (прибыль / активы)")
plt.show()

Облако наклонено вниз: связь есть, и знак осмысленный.

Чтобы не строить такую картинку для каждой пары из 20 показателей, есть **корреляционная матрица**: одно число на пару, от −1 до +1. `.corr()` считает её, `sns.heatmap` раскрашивает.

In [ ]:
RATIOS = ["e_roa", "e_nim", "m_cost_income", "a_npl_gross_loans",
          "a_nco_rate", "c_equity_assets", "l_loans_deposits"]

# .corr() — таблица корреляций всех со всеми
corr = banks[RATIOS].corr()

# annot=True — писать числа в клетках; cmap — цветовая шкала; center=0 — белый цвет на нуле
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1)
plt.title("Корреляции показателей")
plt.show()

Читаем первую строку — с чем связана прибыльность:

* с расходами к доходу — сильно отрицательно;
* с процентной маржой — положительно;
* со списаниями и просрочкой — отрицательно.

Всё это ожидаемо, и это хорошо: если бы знаки были другими, надо было бы искать ошибку в данных, а не открывать новый закон банковского дела.

Последний приём EDA на сегодня — **сравнение групп**. `.groupby()` делит таблицу на части по значению колонки и считает что-нибудь внутри каждой части.

In [ ]:
# failed_next_4q — упал ли банк в ближайшие 4 квартала (True/False)
# .median() — медиана внутри каждой группы (медиана, а не среднее: выбросы её не тянут)
banks.groupby("failed_next_4q")[
    ["c_equity_assets", "a_npl_gross_loans", "e_roa", "m_cost_income"]
].median().round(4)

Разница огромная: перед крахом капитал вдвое тоньше, просрочка в несколько раз выше, прибыль отрицательная. **Сигнал в данных есть** — и весь дальнейший курс про то, как им пользоваться, не обманывая себя.

Пока запомним главное: *посмотреть на данные* — это не одна команда, а четыре вопроса. Что в колонках? Как распределено? Что с выбросами? Что с чем связано?

## 3. Линейная регрессия: одна и та же, но с взаимодействием

Вопрос: **отчего зависит прибыльность банка?**

Возьмём два показателя, которые по смыслу должны на неё влиять:

* `a_nco_rate` — доля кредитов, списанных как безнадёжные (потери по портфелю);
* `c_equity_assets` — капитал к активам (запас прочности).

Модель, которую вы знаете из эконометрики:

$$ \text{ROA} = \beta_0 + \beta_1 \cdot \text{списания} + \beta_2 \cdot \text{капитал} + \varepsilon $$

Считать её будем библиотекой `statsmodels`: она принимает формулу в том же виде, в каком вы её пишете на бумаге.

In [ ]:
# smf — модуль statsmodels, который понимает формулы вида "y ~ x1 + x2"
import statsmodels.formula.api as smf

# ols = ordinary least squares, обычный МНК. Слева от ~ отклик, справа признаки через +.
# .fit() запускает оценивание и возвращает результат
model_simple = smf.ols("e_roa ~ a_nco_rate + c_equity_assets", data=banks).fit()

# .summary().tables[1] — та самая таблица коэффициентов со стандартными ошибками и t
print(model_simple.summary().tables[1])
print("R^2 =", round(model_simple.rsquared, 4))

Знаки осмысленные: списания тянут прибыльность вниз, капитал — вверх. $t$-статистики огромные, но это не заслуга модели: при 95 тысячах наблюдений значимо почти всё. Смотреть надо на **величину**.

Коэффициенты выражены в долях единицы, поэтому переведём их в понятные величины: рост списаний на **1 процентный пункт** — это +0.01 по `a_nco_rate`.

In [ ]:
# .params — коэффициенты модели, обращаемся к нужному по имени
b_nco = model_simple.params["a_nco_rate"]
b_cap = model_simple.params["c_equity_assets"]

# 0.01 — один процентный пункт; 10 000 — перевод доли ROA в базисные пункты
print(f"+1 п.п. списаний ->  {b_nco * 0.01 * 10_000:+.0f} б.п. ROA")
print(f"+1 п.п. капитала ->  {b_cap * 0.01 * 10_000:+.0f} б.п. ROA")

Нарисуем то, что посчитали. `sns.regplot` строит облако точек и проводит через него линию МНК.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.regplot(data=show, x="a_nco_rate", y="e_roa", ax=axes[0],
            scatter_kws={"alpha": 0.2, "s": 10},   # как рисовать точки
            line_kws={"color": "crimson"})          # как рисовать линию
axes[0].set_title("Списания и прибыльность")

sns.regplot(data=show, x="c_equity_assets", y="e_roa", ax=axes[1],
            scatter_kws={"alpha": 0.2, "s": 10}, line_kws={"color": "crimson"})
axes[1].set_title("Капитал и прибыльность")

plt.tight_layout()
plt.show()

### Чего эта модель сказать не может

В формуле выше есть сильное и **незамеченное** утверждение: эффект списаний **одинаков для всех банков**. Один коэффициент $\beta_1$ — одно число на всю выборку. Модель обещает, что процентный пункт списаний стоит одинаково и банку с капиталом 4% активов, и банку с капиталом 20%.

Правдоподобно ли это? Скорее нет. У банка с тонким капиталом списания редко приходят одни: за ними обычно тянутся доначисленные резервы, подорожавшее фондирование и ушедшие клиенты. У банка с толстым капиталом тот же процент списаний — рабочий эпизод.

Проверим глазами: разобьём банки на три группы по капиталу и проведём линию **внутри каждой группы**.

In [ ]:
# pd.qcut делит на 3 равные по числу наблюдений группы (по терцилям) и подписывает их
show = show.copy()
show["капитал"] = pd.qcut(show["c_equity_assets"], 3,
                          labels=["тонкий", "средний", "толстый"])

# lmplot рисует отдельную линию регрессии для каждой группы (hue = "раскрасить по")
sns.lmplot(data=show, x="a_nco_rate", y="e_roa", hue="капитал",
           height=4.5, aspect=1.5, ci=None,
           scatter_kws={"alpha": 0.15, "s": 10})
plt.title("Наклон линии разный: цена списаний зависит от запаса капитала")
plt.xlabel("списания / кредиты")
plt.ylabel("ROA")
plt.show()

Линии не параллельны. У банков с тонким капиталом наклон заметно круче — те же списания сопровождаются большим падением прибыльности.

Именно это и называется **эффектом взаимодействия**: влияние одного признака зависит от значения другого. В формулу он добавляется как произведение:

$$ \text{ROA} = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \beta_3 \cdot (X_1 \times X_2) + \varepsilon $$

В `statsmodels` произведение пишется звёздочкой: `x1 * x2` означает «оба признака и их произведение».

In [ ]:
# Звёздочка = x1 + x2 + x1:x2. Двоеточие — это само произведение.
model_inter = smf.ols("e_roa ~ a_nco_rate * c_equity_assets", data=banks).fit()

print(model_inter.summary().tables[1])
print("R^2 без взаимодействия:", round(model_simple.rsquared, 4))
print("R^2 с взаимодействием: ", round(model_inter.rsquared, 4))

Коэффициент при произведении положителен: чем толще капитал, тем **слабее** удар от списаний. То, что мы видели на картинке, теперь измерено числом.

Но само по себе это число ни о чём не говорит: прочитать «+20.6» невозможно. В модели со взаимодействием предельный эффект перестаёт быть одним числом и становится формулой:

$$ \frac{\partial \text{ROA}}{\partial \, \text{списания}} = \beta_1 + \beta_3 \cdot \text{капитал} $$

Вот её и считают.

In [ ]:
b1 = model_inter.params["a_nco_rate"]                     # эффект при нулевом капитале
b3 = model_inter.params["a_nco_rate:c_equity_assets"]     # добавка за каждую единицу капитала

print("сколько ROA стоит рост списаний на 1 п.п.:")
for q in [0.1, 0.5, 0.9]:
    cap = banks["c_equity_assets"].quantile(q)            # значение капитала на перцентиле q
    effect = (b1 + b3 * cap) * 0.01 * 10_000              # предельный эффект в базисных пунктах
    print(f"  капитал/активы = {cap:.3f} (перцентиль {int(q * 100):2d}):  {effect:+6.0f} б.п.")

print(f"\nмодель без взаимодействия обещает всем одинаковые "
      f"{b_nco * 0.01 * 10_000:+.0f} б.п.")

Вот зачем это нужно. У тонко капитализированного банка процентный пункт списаний идёт вместе с падением ROA на 226 базисных пунктов, у хорошо капитализированного — на 83. Разница почти втрое. Модель без взаимодействия даёт всем одну цифру посередине: она **систематически недооценивает риск у слабых банков** и переоценивает у сильных.

### Что мы измерили и чего не измерили

Мы измерили, что **наклон разный**. Мы не измерили, **почему** он разный. Объяснений как минимум два:

1. состав баланса — у банков с разным капиталом разная доля кредитов в активах, и тот же процент списаний по портфелю даёт разный процент по активам;
2. списания у слабо капитализированного банка приходят «в компании» с резервами, дорогим фондированием и оттоком клиентов, и мы видим суммарный эффект всего этого сразу.

Различить эти истории данными лекции 1 нельзя. Поэтому честная формулировка результата — «у банков с тонким капиталом списания **сопровождаются** большим падением прибыльности». Формулировка «толстый капитал **защищает** прибыль» — уже причинное утверждение, и его мы не показали. Разницу между этими двумя фразами вас будут спрашивать весь курс.

## 4. Регрессия против ближайших соседей

Линейная регрессия делает два шага:

1. **предполагает форму связи** — прямая (или плоскость);
2. подбирает несколько чисел так, чтобы эта прямая села на данные как можно лучше.

Первый шаг — сильное допущение. Если настоящая связь не прямая, никакой подбор коэффициентов не спасёт.

Есть модели, которые формы не предполагают вообще. Самая простая — **метод $k$ ближайших соседей** (KNN): чтобы предсказать ROA банка, найдём $k$ самых похожих на него банков и возьмём их средний ROA. Никакой формулы, только данные.

Сравним оба подхода на одной задаче: **предсказать ROA по доле просроченных кредитов**.

Сравнивать будем честно: обучим модели на одной части данных, а ошибку посчитаем на другой, которую они не видели. Это называется **обучающая** и **тестовая** выборки, и это главная гигиеническая привычка курса.

In [ ]:
# Из библиотеки sklearn берём ровно те инструменты, которые нужны
from sklearn.model_selection import train_test_split     # деление на обучение/тест
from sklearn.linear_model import LinearRegression        # линейная регрессия
from sklearn.neighbors import KNeighborsRegressor        # k ближайших соседей
from sklearn.metrics import mean_squared_error           # ошибка предсказания

# sklearn не умеет работать с пропусками — выбрасываем строки, где нет нужных значений
model_df = banks.dropna(subset=["a_npl_gross_loans", "e_roa"])

# X — признаки (в двойных скобках: sklearn ждёт таблицу, а не одну колонку)
# y — то, что предсказываем
X = model_df[["a_npl_gross_loans"]]
y = model_df["e_roa"]

# 70% строк на обучение, 30% на тест. random_state — фиксация случайности.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=SEED)

print("обучающая выборка:", len(X_train), "строк")
print("тестовая выборка: ", len(X_test), "строк")

In [ ]:
# Обучаем обе модели на ОДНИХ И ТЕХ ЖЕ обучающих данных.
# .fit() — обучить, .predict() — предсказать.
linreg = LinearRegression().fit(X_train, y_train)
knn = KNeighborsRegressor(n_neighbors=50).fit(X_train, y_train)

# RMSE — корень из средней квадратичной ошибки, в тех же единицах, что и ROA.
# Умножаем на 10 000, чтобы читать в базисных пунктах.
def rmse(model, X, y):
    return np.sqrt(mean_squared_error(y, model.predict(X))) * 10_000

pd.DataFrame({
    "на обучающей": [rmse(linreg, X_train, y_train), rmse(knn, X_train, y_train)],
    "на тестовой":  [rmse(linreg, X_test, y_test),   rmse(knn, X_test, y_test)],
}, index=["линейная регрессия", "KNN, k = 50"]).round(1)

Числа — ошибка предсказания ROA в базисных пунктах: меньше значит лучше.

Читаем по столбцам, а не по строкам. У линейной регрессии обе ошибки почти одинаковы: модель настолько жёсткая, что ей нечем «подстроиться» под конкретную выборку. У KNN ошибка на обучающей заметно меньше, чем на тестовой, — вот эту разницу и называют переобучением. Выигрыш на тесте у KNN есть, но скромный: пара процентов.

Посмотрим на обе модели глазами. Построим сетку значений просрочки и спросим каждую модель, какой ROA она предсказывает в каждой точке.

In [ ]:
# Сетка: 200 точек от 0 до 15% просрочки — там живёт 99% банк-кварталов
grid = pd.DataFrame({"a_npl_gross_loans": np.linspace(0, 0.15, 200)})

sns.scatterplot(data=show, x="a_npl_gross_loans", y="e_roa", alpha=0.2, s=10, color="grey")
plt.plot(grid, linreg.predict(grid), color="crimson", lw=3, label="линейная регрессия")
plt.plot(grid, knn.predict(grid), color="steelblue", lw=3, label="KNN, k = 50")
plt.xlim(0, 0.15)
plt.title("Одна прямая против средних по 50 соседям")
plt.xlabel("просрочка / кредиты")
plt.ylabel("ROA")
plt.legend()
plt.show()

Красная прямая обязана быть прямой — такова форма, которую мы ей задали: один и тот же наклон на всём диапазоне, от здоровых банков до предбанкротных.

Синяя линия ничего не обязана, и видно две вещи. До 4% просрочки она почти совпадает с прямой: там банков много, и связь действительно близка к линейной. Дальше она уходит **ниже** прямой — то есть при высокой просрочке прибыльность падает круче, чем обещает линейная модель, — и одновременно начинает дёргаться. Дёргается она не потому, что там что-то происходит, а потому, что каждое соседство собрано из немногих наблюдений: это уже не сигнал, а шум.

Возникает соблазн: раз гибкость помогает, давайте сделаем модель ещё гибче. Уменьшим $k$ — пусть усредняет не по 50 соседям, а по одному.

In [ ]:
# Цикл по разным k: каждый раз обучаем модель заново и считаем обе ошибки
rows = []
for k in [1, 5, 50, 500, 5000]:
    m = KNeighborsRegressor(n_neighbors=k).fit(X_train, y_train)
    rows.append({"соседей k": k,
                 "на обучающей": rmse(m, X_train, y_train),
                 "на тестовой":  rmse(m, X_test, y_test)})

pd.DataFrame(rows).set_index("соседей k").round(1)

Прочитайте таблицу внимательно — в ней половина курса.

* **$k = 1$: ошибка на обучающей выборке втрое меньше, чем у всех остальных.** Каждый банк сам себе ближайший сосед, поэтому модель почти в точности воспроизводит данные, на которых училась. И при этом на новых данных она **худшая из всех**, хуже прямой. Мы аккуратно выучили не только сигнал, но и случайный шум.
* **$k = 5000$: другая крайность.** Усредняем такую широкую окрестность, что различия между банками стираются, и ошибка снова растёт.
* **Минимум посередине**, в районе нескольких сотен соседей.

Это **переобучение** (слева) и **недообучение** (справа) на одной таблице. Отсюда главный вывод сегодняшнего дня:

> «Модель идеально села на данные» — не комплимент. Единственная проверка, которой можно верить, — ошибка на данных, которых модель не видела.

## Итог

1. Данные читаются одной командой, но **строка данных всегда что-то значит**: у нас это банк в квартале.
2. «Посмотреть на данные» — это `.info()`, `.describe()`, гистограммы и корреляции. Выбросы находятся глазами, а выбрасываются **явно**, вслух и с оговоркой о том, кого мы теперь описываем.
3. **Взаимодействие** — это «эффект $X_1$ зависит от $X_2$». Читается оно не по коэффициенту при произведении, а по предельному эффекту на разных значениях $X_2$.
4. **«Наклон разный» и «одно влияет на другое» — разные утверждения.** Первое мы измерили, второе — нет.
5. **Гибкая модель не значит лучшая.** Сравнивать модели можно только на данных, которых они не видели.

---

# Домашнее задание

Сдаётся в этом же ноутбуке: сделайте свою копию, заполните ячейки, помеченные `TODO(hw1-…)`, и пришлите файл. Пояснения пишите по-русски, **код и имена переменных — по-английски**.

Задание — те же четыре шага, но на **другом отклике**: в лекции мы объясняли `e_roa` (прибыльность), вы будете объяснять **`e_nim` — процентную маржу**: сколько процентного дохода банк получает на доллар активов.

Копировать код из лекции можно и нужно — менять придётся немногое. Смысл задания не в том, чтобы написать новый код, а в том, чтобы **выбрать** признаки, фильтр и $k$ — и уметь объяснить каждый выбор.

### 1. Данные

`TODO(hw1-a)`. Загрузите `banks_panel.parquet` в переменную `hw` и напечатайте: число строк, число банков, период. В ячейке ниже, словами: **что такое одна строка этой таблицы** и почему строк намного больше, чем банков.

In [ ]:
# TODO(hw1-a): загрузите панель в переменную `hw` и напечатайте её размер, число банков и период.
raise NotImplementedError

> _Ваш ответ (2–3 предложения):_

### 2. Посмотреть на данные

`TODO(hw1-b)`. Выберите **четыре** показателя, которые по вашей гипотезе связаны с процентной маржой, и постройте для них гистограммы (одна ячейка, цикл `for`, как в §2.3). Затем постройте корреляционную матрицу этих четырёх показателей вместе с `e_nim`.

Ниже словами: одно наблюдение, которое вас **удивило**, и одно, которое подтвердило ожидания.

In [ ]:
# TODO(hw1-b): четыре гистограммы одним циклом + тепловая карта корреляций с e_nim.
raise NotImplementedError

> _Ваш ответ (2–3 предложения):_

### 3. Чистка

`TODO(hw1-c)`. Отфильтруйте панель в переменную `hw_clean`: условия выбираете сами, но каждое **должно быть прокомментировано** — почему именно эта граница. Напечатайте, сколько строк было и сколько осталось.

Ниже одним предложением: **о какой совокупности банков** теперь ваши выводы.

In [ ]:
# TODO(hw1-c): постройте hw_clean явным фильтром; каждое условие с комментарием.
# Напечатайте: было строк / осталось строк / осталось банков.
raise NotImplementedError

> _Ваш ответ (1 предложение):_

### 4. Две регрессии

`TODO(hw1-d)`. Выберите два признака $X_1$ и $X_2$ из своих четырёх и оцените на `hw_clean`:

* `m_simple` — модель `e_nim ~ X1 + X2`;
* `m_inter` — та же модель со взаимодействием (`X1 * X2`).

Напечатайте таблицы коэффициентов, оба $R^2$ и **предельный эффект** $X_1$ на 10-м, 50-м и 90-м перцентилях $X_2$.

Ниже, ~120 слов: что показывает взаимодействие, и — отдельным предложением — какое утверждение вы **не** имеете права сделать по этим оценкам.

In [ ]:
# TODO(hw1-d): две модели statsmodels — `m_simple` и `m_inter` — на hw_clean,
# отклик e_nim. Напечатайте коэффициенты, R^2 обеих и предельный эффект X1
# на перцентилях 10 / 50 / 90 переменной X2.
raise NotImplementedError

> _Ваш ответ (~120 слов):_

### 5. Регрессия против соседей

`TODO(hw1-e)`. Разбейте `hw_clean` на обучающую и тестовую выборки (30% на тест, `random_state=SEED`). Предскажите `e_nim` по **одному** признаку двумя моделями: линейной регрессией и KNN при $k \in \{1, 5, 50, 500, 5000\}$. Соберите ошибки в таблицу `scores` (строки — модели, столбцы — обучающая и тестовая RMSE) и постройте картинку с обеими подгонками, как в §4.

Ниже, ~80 слов: какое $k$ вы бы выбрали и почему **не** $k = 1$, хотя на обучающей выборке он лучший.

In [ ]:
# TODO(hw1-e): train/test split, LinearRegression и KNN при разных k,
# таблица `scores` с обучающей и тестовой RMSE + график с двумя подгонками.
raise NotImplementedError

> _Ваш ответ (~80 слов):_

### Самопроверка

Запустите ячейку ниже после того, как заполните все заглушки. Зелёный прогон — необходимое, но **не достаточное** условие: смысл ответов оценивает человек, а не `assert`.

In [ ]:
assert isinstance(hw, pd.DataFrame) and len(hw) > 0, "hw1-a: `hw` пуста или не DataFrame"
assert isinstance(hw_clean, pd.DataFrame), "hw1-c: `hw_clean` должен быть DataFrame"
assert len(hw_clean) < len(hw), "hw1-c: фильтр ничего не отсёк — так не бывает"
assert hasattr(m_simple, "params") and hasattr(m_inter, "params"), \
    "hw1-d: ожидаются две оценённые модели statsmodels"
assert len(m_inter.params) > len(m_simple.params), \
    "hw1-d: в модели со взаимодействием должно быть больше коэффициентов"
assert "e_nim" in m_inter.model.formula, "hw1-d: отклик должен быть e_nim"
assert isinstance(scores, pd.DataFrame) and scores.shape[0] >= 2, \
    "hw1-e: `scores` — таблица минимум с двумя моделями"
print("OK · формальная проверка пройдена")